# DeBERTa v3 Resume NER Training
## Training on Experience and Education Entity Extraction

This notebook trains a DeBERTa v3 model for Named Entity Recognition on resume data.

**Dataset:** 19,292 training samples (15,433 train, 1,930 test)

**Labels:** 25 entity types for Work Experience and Education sections

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets seqeval scikit-learn accelerate sentencepiece

In [ ]:
# Import libraries
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Upload Training Data

Upload these files from your local machine:
- `simple_dataset_train.conll`
- `simple_dataset_test.conll`

Or mount Google Drive if you've uploaded them there.

In [ ]:
# Option 1: Mount Google Drive (if files are in Drive)
from google.colab import drive
drive.mount('/content/drive')

# Update these paths to where your files are in Drive
# train_file = "/content/drive/MyDrive/resume-ner/simple_dataset_train.conll"
# test_file = "/content/drive/MyDrive/resume-ner/simple_dataset_test.conll"

In [ ]:
# Option 2: Upload files directly
from google.colab import files

print("Upload simple_dataset_train.conll:")
uploaded = files.upload()
train_file = list(uploaded.keys())[0]

print("\nUpload simple_dataset_test.conll:")
uploaded = files.upload()
test_file = list(uploaded.keys())[0]

print(f"\n✅ Files uploaded: {train_file}, {test_file}")

## 3. Define Labels and Helper Functions

In [ ]:
# Define all entity labels
LABELS = [
    'O',
    'B-COMPANY', 'I-COMPANY',
    'B-CLIENT', 'I-CLIENT',
    'B-ROLE', 'I-ROLE',
    'B-LOCATION', 'I-LOCATION',
    'B-DATE_START', 'I-DATE_START',
    'B-DATE_END', 'I-DATE_END',
    'B-DEGREE', 'I-DEGREE',
    'B-FIELD', 'I-FIELD',
    'B-INSTITUTION', 'I-INSTITUTION',
    'B-EDU_YEAR_START', 'I-EDU_YEAR_START',
    'B-EDU_YEAR_END', 'I-EDU_YEAR_END',
    'B-GRADE', 'I-GRADE'
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for i, label in enumerate(LABELS)}

print(f"Total labels: {len(LABELS)}")
print(f"Labels: {LABELS}")

In [ ]:
def read_conll_file(file_path):
    """Read CoNLL format file and return sentences with tokens and labels."""
    sentences = []
    current_tokens = []
    current_labels = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if line:  # Non-empty line
                parts = line.split('\t')
                if len(parts) == 2:
                    token, label = parts
                    current_tokens.append(token)
                    current_labels.append(label)
            else:  # Empty line = sentence boundary
                if current_tokens:
                    sentences.append({
                        'tokens': current_tokens,
                        'ner_tags': current_labels
                    })
                    current_tokens = []
                    current_labels = []
    
    # Add last sentence if file doesn't end with empty line
    if current_tokens:
        sentences.append({
            'tokens': current_tokens,
            'ner_tags': current_labels
        })
    
    return sentences

## 4. Load and Prepare Datasets

In [ ]:
# Load CoNLL files
print("📖 Loading training data...")
train_sentences = read_conll_file(train_file)
print(f"✅ Loaded {len(train_sentences)} training sentences")

print("\n📖 Loading test data...")
test_sentences = read_conll_file(test_file)
print(f"✅ Loaded {len(test_sentences)} test sentences")

# Show sample
print("\n📝 Sample sentence:")
sample = train_sentences[0]
print(f"Tokens: {sample['tokens'][:10]}")
print(f"Labels: {sample['ner_tags'][:10]}")

In [ ]:
# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_list(train_sentences)
test_dataset = Dataset.from_list(test_sentences)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Test dataset: {len(test_dataset)} examples")

## 5. Load Tokenizer and Model

In [ ]:
# Load tokenizer
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"✅ Loaded tokenizer: {model_name}")

In [ ]:
# Tokenization function with label alignment
def tokenize_and_align_labels(examples, max_length=1024):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=max_length,
        padding=False
    )
    
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

In [ ]:
# Tokenize datasets
print("🔤 Tokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)

print("✅ Tokenization complete!")

In [ ]:
# Load model
print("🤖 Loading model...")
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Model loaded with {len(LABELS)} labels")

## 6. Define Metrics and Training Arguments

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    return {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions),
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./resume-ner-deberta",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=500,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    gradient_checkpointing=True,
    report_to="none"
)

print("✅ Training arguments configured")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   FP16: {training_args.fp16}")

## 7. Initialize Trainer and Start Training

In [ ]:
# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("✅ Trainer initialized")

In [ ]:
# Start training
print("🏋️ Starting training...")
print("This will take approximately 2-3 hours with GPU")
print("="*60)

trainer.train()

## 8. Evaluate and Save Model

In [ ]:
# Final evaluation
print("📊 Final evaluation...")
results = trainer.evaluate()

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
for key, value in results.items():
    print(f"{key}: {value:.4f}")
print("="*60)

In [ ]:
# Save model
output_dir = "./resume-ner-improved"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✅ Model saved to: {output_dir}")

# Save label mappings
import json
with open(f"{output_dir}/label_mappings.json", 'w') as f:
    json.dump({
        'labels': LABELS,
        'label2id': label2id,
        'id2label': {str(k): v for k, v in id2label.items()}
    }, f, indent=2)

print("✅ Label mappings saved")

## 9. Download Trained Model

In [ ]:
# Zip the model directory
!zip -r resume-ner-improved.zip ./resume-ner-improved

# Download the zip file
from google.colab import files
files.download('resume-ner-improved.zip')

print("✅ Model downloaded! Extract and place in your ai-service/models/ directory")

## 10. Test Inference (Optional)

In [ ]:
# Test the model with a sample
from transformers import pipeline

# Create NER pipeline
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

# Test text
test_text = "John Smith worked at Google as Software Engineer from Jan 2020 to Dec 2022 in Mountain View"

# Get predictions
predictions = ner_pipeline(test_text)

print("\n📝 Test Inference:")
print(f"Input: {test_text}")
print("\nPredictions:")
for pred in predictions:
    print(f"  {pred['word']}: {pred['entity_group']} (score: {pred['score']:.3f})")